In [ ]:
import psycopg2

In [ ]:
import os
print(os.getcwd())

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(
    host=os.environ["DB_HOST"],
    port=os.environ["DB_PORT"],
    dbname=os.environ["DB_NAME"],
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"]
)

In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT * FROM RxOrders LIMIT 10;")
    rows = cur.fetchall()

    for row in rows:
        print(row)

In [ ]:
#check column names of RxOrders
with conn.cursor() as cur:
    cur.execute("SELECT * FROM RxOrders LIMIT 0;")
    print([col[0] for col in cur.description])

In [ ]:
#check column names of RxOrderHistory
with conn.cursor() as cur:
    cur.execute("SELECT * FROM RxOrderHistory LIMIT 0;")
    print([col[0] for col in cur.description])

In [ ]:
#check column names of RxPacked
with conn.cursor() as cur:
    cur.execute("SELECT * FROM RxPacked LIMIT 0;")
    print([col[0] for col in cur.description])

In [ ]:
#check column names of RxWritten
with conn.cursor() as cur:
    cur.execute("SELECT * FROM RxWritten LIMIT 0;")
    print([col[0] for col in cur.description])

In [ ]:
#check column names of RxShipments
with conn.cursor() as cur:
    cur.execute("SELECT * FROM RxShipments LIMIT 0;")
    print([col[0] for col in cur.description])

In [ ]:
#check column names of RxMatches
with conn.cursor() as cur:
    cur.execute("SELECT * FROM RxMatches LIMIT 0;")
    print([col[0] for col in cur.description])

In [ ]:
#check row count
with conn.cursor() as cur:
    
    cur.execute("SELECT COUNT(*) FROM RxOrders;")
    print("RxOrders:", cur.fetchone()[0])
    
with conn.cursor() as cur:
    
    cur.execute("SELECT COUNT(*) FROM RxOrderHistory;")
    print("RxOrderHistory:", cur.fetchone()[0])
    

with conn.cursor() as cur:
    
    cur.execute("SELECT COUNT(*) FROM RxPacked;")
    print("RxPacked:", cur.fetchone()[0])

with conn.cursor() as cur:
    
    cur.execute("SELECT COUNT(*) FROM RxWritten;")
    print("RxWritten:", cur.fetchone()[0])
    
with conn.cursor() as cur:
    
    cur.execute("SELECT COUNT(*) FROM RxShipments;")
    print("RxShipments:", cur.fetchone()[0])
    
with conn.cursor() as cur:
    
    cur.execute("SELECT COUNT(*) FROM RxMatches;")
    print("RxMatches:", cur.fetchone()[0])

In [ ]:
conn.rollback()

In [ ]:
#convert to python df
import pandas as pd

with conn.cursor() as cur:
    cur.execute("SELECT * FROM RxOrders;")
    rows = cur.fetchall()
    columns = [col[0] for col in cur.description]

rx_orders = pd.DataFrame(rows, columns=columns)

print(rx_orders.head())
print(rx_orders.shape)  # (rows, columns)

In [ ]:
rx_orders.columns

In [ ]:
rx_orders.isna().sum()

In [ ]:
rx_orders.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.hist(rx_orders['refills_allowed'].dropna(), bins = np.arange(0,45, 2), edgecolor = "black")
plt.xlim(0,40)
plt.xlabel('Number of Refills')
plt.ylabel("Number of Patients")

plt.savefig("EDA/hist_refills.png", dpi = 300, bbox_inches = "tight")
plt.show()


In [ ]:

print(max(rx_orders['refills_allowed']))

In [ ]:
plt.boxplot(rx_orders['refills_allowed'].dropna())
plt.ylabel("Number of Refills")
plt.savefig("EDA/boxplot_refills.png", dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
rx_orders['quantity'].head()

In [ ]:
import seaborn as sns
plot=sns.histplot(
    data = rx_orders,
    x="quantity", 
    hue = "bc_prescribed"
)
plot.set(
    xlabel = "Quantity",
    ylabel="Person Count",
    title = "Quantity Grouped by Birth control or not"
)

sns.move_legend(
    
    plot, 
    "upper right",
    title = "Birth Control Prescribed"
)
plt.savefig("EDA/quantity_by_bc.png", dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:

bc=sns.histplot(data = rx_orders, x='bc_prescribed', discrete = True, stat = 'count')
plt.xlabel("Birth Control Prescribed")
plt.title("Was Birth Control Prescribed?")
bc.set_xticks((0,1))
plt.savefig("EDA/bc_prescribed.png", dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
rx_orders.groupby("bc_prescribed")["quantity"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)


In [ ]:
rx_orders['sku'].unique()

In [ ]:
rx_orders.groupby("sku")["quantity"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

In [ ]:
type=sns.histplot(data = rx_orders, x='sku', discrete = True, stat = 'count', color='green')
plt.xlabel("Medication Prescribed")
plt.title("Medication Type Count")
plt.xticks(rotation=45)
#type.set_xticks((0,1))
plt.savefig("EDA/medication_type.png", dpi = 300, bbox_inches = "tight")
plt.show()